In [3]:
import os
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

# Base URL for the API
BASE_URL = "https://goat.genomehubs.org/api/v2/search?query="

def log_message(message, logfile):
    with open(logfile, "a") as log:
        log.write(message + "\n")

# Get today's date
today = datetime.date.today()
logfile = f"Overlaps_GoaT_Projects_ALL_{today}.txt"
log_message(f"Overlaps search of GoaT {today}", logfile)

# Read project data
project_info_all = pd.read_csv("GoaT_Projects.tsv", sep="\t")
goat_projects = project_info_all['goat_project'].tolist()
sequencing_statuses = project_info_all['sequencing_status'].tolist()
contact_emails = project_info_all['contact_email'].tolist()
contact_names = project_info_all['contact_name'].tolist()

# Create all-by-all matrix
row_names = goat_projects
col_names = goat_projects
overlap_counts = pd.DataFrame(0, index=row_names, columns=col_names)

# Loops to cycle through all pairs of projects
for p1, project1 in enumerate(goat_projects):
    for p2, project2 in enumerate(goat_projects):
        if project1 == project2:
            continue

        p1seq = sequencing_statuses[p1]
        p2seq = sequencing_statuses[p2]
        p1cem = contact_emails[p1].split(';') if pd.notna(contact_emails[p1]) else []
        p2cem = contact_emails[p2].split(';') if pd.notna(contact_emails[p2]) else []
        p1cna = contact_names[p1]

        # Build the API query
        mainq = (f"long_list%3D{project1}%20AND%20tax_rank(species)%20AND%20"
                 f"{p2seq}%3D%21null&result=taxon&includeEstimates=true&taxonomy=ncbi"
                 f"&fields={p1seq}%2C{p2seq}&size=10000")
        query = BASE_URL + mainq
        response = requests.get(query)
        ovlap = response.json()

        if ovlap.get("status", {}).get("hits", 0) == 0:
            log_message(f"No overlaps detected between {project1} & {project2}", logfile)
            continue

        # Save results to a file
        specs = pd.json_normalize(ovlap["results"]["result"])
        outfile = f"Overlap_{project1}_{project2}_{today}.tsv"

        with open(outfile, "w") as f:
            f.write("scientific_name\ttaxon_id\t{}\t{}\tGoat-URL\n".format(project1, project2))
            for _, row in specs.iterrows():
                sname = row.get("scientific_name", "")
                taxid = row.get("taxon_id", "")
                p1stat = row.get(f"fields.{p1seq}.value", "long_list")
                p2stat = row.get(f"fields.{p2seq}.value", "long_list")
                goaturl = f"https://goat.genomehubs.org/search?query=tax_name%28{taxid}%29&result=taxon&includeEstimates=true&taxonomy=ncbi"

                f.write(f"{sname}\t{taxid}\t{p1stat}\t{p2stat}\t{goaturl}\n")

        # Read, sort, and overwrite results file
        textblob = pd.read_csv(outfile, sep="\t")
        textsort = textblob.sort_values(by=[project1, project2])
        textsort.to_csv(outfile, sep="\t", index=False)

        longlist = len(textsort)
        log_message(f"{longlist} overlaps detected between {project1} & {project2}", logfile)
        overlap_counts.loc[project1, project2] = longlist

# Generate heatmap
pdfout = f"Overlaps_GoaT_Projects_ALL_{today}.pdf"
plottitle = f"GoaT Project Overlaps {today} (max={overlap_counts.max().max()})"

plt.figure(figsize=(12, 12))
plt.imshow(np.log1p(overlap_counts), cmap="hot", interpolation="nearest")
plt.colorbar()
plt.title(plottitle)
plt.xticks(ticks=range(len(col_names)), labels=col_names, rotation=90)
plt.yticks(ticks=range(len(row_names)), labels=row_names)
plt.tight_layout()
plt.savefig(pdfout)
plt.close()

log_message(f"Heatmap saved to {pdfout}", logfile)


KeyboardInterrupt: 